# 独立干扰与有效变化
先对 heldout 草稿做可缓存的机器审核，再评测；人工抽查另行记录。禁止用于训练或调参。

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'full'  # Change to 'full' only after the pilot completes and memory is measured.
ROOT = DRIVE / 'public_lightrag_v1' / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
PUBLIC_ASSETS = DRIVE / 'public_lightrag_assets'
CONFIG_PATH = PROJECT / 'configs/public_lightrag.yaml'
print('Code:', PROJECT, 'Artifacts:', ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/graphrag-jev Artifacts: /content/drive/MyDrive/SA-Jev/public_lightrag_v1/full


In [2]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    # BF16 + LoRA does not use torchao quantization. Colab's old optional
    # torchao package makes PEFT 0.19.1 fail even on ordinary Linear layers.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config(CONFIG_PATH)
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), '--config', str(CONFIG_PATH), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


SA-Jev source: /content/graphrag-jev/src/sa_jev/__init__.py


In [3]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


Credential file: /content/drive/MyDrive/SA-Jev/.env
OPENAI_API_KEY: loaded
OPENAI_BASE_URL: loaded
TYPESAFE_API_KEY: loaded
HF_TOKEN: loaded


先审核已有 heldout 扰动草稿，再评测。审核通过表示机器审核，不代表人工核验或绝对语义保证。默认只预检查 3 个包；确认 API 返回正常后设置 REVIEW_LIMIT=None。预检查不写正式测试文件，不运行模型评测。

In [ ]:
from pathlib import Path
import importlib
from sa_jev import context_views, perturbation_review
importlib.reload(context_views)
importlib.reload(perturbation_review)

# 保持 MODE='full' 使用已经准备好的数据；不要为了少量检查切换 ROOT。
REVIEW_LIMIT = 3  # 首次预检查；随后改为 None，审核全部草稿并导出正式文件。
config.setdefault('perturbation_review', {}).update(max_tokens=4096, disable_thinking=True)
data_dir = ROOT / 'data'
draft_path = data_dir / 'perturbations_draft.jsonl'
if not draft_path.exists():
    cli('review-packets')
print('草稿:', draft_path)
print('审核范围:', '全部草稿' if REVIEW_LIMIT is None else f'前 {REVIEW_LIMIT} 个包（预检查）')

In [ ]:
review_summary = perturbation_review.prepare_perturbation_reviews(
    ROOT, config, limit=REVIEW_LIMIT)
print({k: v for k, v in review_summary.items() if k != 'decisions'})

审核中断后重跑审核格，会复用逐包缓存。API 输出不完整时停止并保留原始响应；不伪造审核通过。全部审核后，data/perturbations_reviewed.jsonl 保存合格包，配套 manifest 记录实际数量、通过率和拒绝原因；perturbation_reviews/ 保存原文、Gold 和 API 审核记录（仅审核使用，不输入评测模型）。perturbations_reviewed.human_spotcheck.json 是待填写的人工抽查清单，不代表人工审核已完成。论文需报告机器审核及真实人工抽查结果；若发现错误，应修正或移除对应包并更新评测集记录，不能用于选择模型或调参。

In [ ]:
from sa_jev.experiments import robustness_experiment
if REVIEW_LIMIT is not None:
    print('预检查完成，尚未运行正式评测。将 REVIEW_LIMIT 改为 None，重跑设置格、审核格及本格。')
else:
    from sa_jev.io import read_jsonl, digest
    reviewed_path = ROOT / 'data/perturbations_reviewed.jsonl'
    if (review_summary['scope'] != 'full_draft' or review_summary['status'] != 'complete'
            or digest(list(read_jsonl(reviewed_path))) != review_summary['output_hash']):
        raise ValueError('正式审核导出不完整或已变更，请检查审核记录。')
    rows = robustness_experiment(ROOT, config)

同时检查 JS、Top-1/Top-5、充分性翻转、错误提前停止、nDCG 和有效变化后正确补证。新增已选证据会改变剩余候选集合，不强行比较为不变分布。独立扰动测试禁止用于训练或调参。